# 🧬 Dark Manifold v6 — Hybrid Biochemical Neural Model

## Core Principle

**Physics where we know it, learning where we don't.**

| Component | Source | Type |
|-----------|--------|------|
| Stoichiometry S | reconstruction.xlsx | FIXED |
| Known Km values | BRENDA/literature | FIXED |
| MM kinetics equation | Biochemistry | FIXED |
| Unknown Km values | Neural network | LEARNED |
| Gene→Enzyme mapping | Neural network | LEARNED |
| Regulatory effects | Neural network | LEARNED |

## Key Innovation

No separate generator! The model IS the simulation.
Train with physics-based losses (energy conservation, mass balance, stability).

In [ ]:
#@title 1️⃣ Setup & Upload Data
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Real Biochemistry Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}
name_to_idx = {n.lower(): i for i, n in enumerate(met_names)}

# Parse genes
genes = set()
gene_to_rxns = {}
for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry matrix (FIXED - real biochemistry)
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation'].lower()
    for i, name in enumerate(met_names):
        name_lower = name.lower()
        if name_lower in equation:
            left = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[i, j] = -1 if name_lower in left else +1

# Build gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

print(f"Genes: {n_genes}, Metabolites: {n_mets}, Reactions: {n_rxns}")
print(f"Stoichiometry entries: {np.count_nonzero(S)}")
print(f"Gene-reaction links: {np.count_nonzero(G)}")

In [ ]:
#@title 3️⃣ Known Kinetic Parameters (from BRENDA/Literature)

# Real Km values from Thornburg et al. (2022) and BRENDA
# These are FIXED - we don't learn them
KNOWN_KM = {
    # Glycolysis (values in mM)
    'glucose': 0.015,
    'g6p': 0.3,
    'f6p': 0.1,
    'fbp': 0.02,
    'g3p': 0.05,
    'dhap': 0.5,
    'bpg13': 0.003,
    'pg3': 0.2,
    'pg2': 0.065,
    'pep': 0.3,
    'pyruvate': 0.16,
    # Nucleotides
    'atp': 0.1,
    'adp': 0.35,
    'amp': 0.11,
    # Cofactors
    'nad': 0.045,
    'nadh': 0.01,
    'nadp': 0.007,
    'nadph': 0.005,
}

# Initial concentrations (mM) - from Park et al. scaled for syn3A
INITIAL_CONC = {
    'atp': 4.0, 'adp': 0.5, 'amp': 0.1,
    'gtp': 0.5, 'gdp': 0.05,
    'nad': 2.0, 'nadh': 0.08,
    'nadp': 0.002, 'nadph': 0.12,
    'glucose': 0.5, 'g6p': 0.2, 'f6p': 0.08,
    'pyruvate': 0.4, 'lactate': 0.5,
    'pi': 10.0, 'ppi': 0.1,
}

# Build Km vector with known values fixed
known_km_indices = []
known_km_values = []

for i, met_name in enumerate(met_names):
    met_lower = met_name.lower()
    for key, value in KNOWN_KM.items():
        if key in met_lower:
            known_km_indices.append(i)
            known_km_values.append(value)
            break

print(f"Known Km values: {len(known_km_indices)} metabolites")
print(f"Unknown Km values: {n_mets - len(known_km_indices)} metabolites (will be learned)")

In [ ]:
#@title 4️⃣ Physics-Based Losses

class PhysicsLoss(nn.Module):
    """
    Loss functions based on physical constraints.
    No external generator needed - the physics IS the supervision.
    """
    
    def __init__(self, S, atp_idx, adp_idx, amp_idx):
        super().__init__()
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.atp_idx = atp_idx
        self.adp_idx = adp_idx
        self.amp_idx = amp_idx
    
    def energy_conservation(self, met_conc, met_next):
        """
        ATP + ADP + AMP should be roughly constant (adenine pool).
        """
        pool_now = met_conc[:, self.atp_idx] + met_conc[:, self.adp_idx] + met_conc[:, self.amp_idx]
        pool_next = met_next[:, self.atp_idx] + met_next[:, self.adp_idx] + met_next[:, self.amp_idx]
        return F.mse_loss(pool_next, pool_now)
    
    def mass_balance(self, flux):
        """
        Net production/consumption should balance for internal metabolites.
        dM/dt = S @ flux should be bounded (not exploding).
        """
        dM = flux @ self.S.T
        return (dM ** 2).mean()  # Penalize large changes
    
    def stability(self, met_conc, met_next):
        """
        Concentrations should stay positive and bounded.
        """
        # Penalize negative concentrations
        neg_penalty = F.relu(-met_next).mean()
        
        # Penalize explosions (>100 mM is unrealistic)
        explosion_penalty = F.relu(met_next - 100).mean()
        
        # Penalize collapse (total pool < 10% of initial)
        total_now = met_conc.sum(dim=-1)
        total_next = met_next.sum(dim=-1)
        collapse_penalty = F.relu(0.1 * total_now - total_next).mean()
        
        return neg_penalty + explosion_penalty + collapse_penalty
    
    def smoothness(self, met_conc, met_next):
        """
        Changes should be gradual, not jumpy.
        """
        delta = met_next - met_conc
        return (delta ** 2).mean()
    
    def energy_charge_valid(self, met_conc):
        """
        Energy charge = (ATP + 0.5*ADP) / (ATP + ADP + AMP)
        Should be between 0.8 and 0.95 for healthy cells.
        """
        atp = met_conc[:, self.atp_idx]
        adp = met_conc[:, self.adp_idx]
        amp = met_conc[:, self.amp_idx]
        
        ec = (atp + 0.5 * adp) / (atp + adp + amp + 1e-6)
        
        # Penalize EC outside healthy range
        low_penalty = F.relu(0.8 - ec).mean()
        high_penalty = F.relu(ec - 0.95).mean()
        
        return low_penalty + high_penalty
    
    def forward(self, met_conc, met_next, flux):
        """
        Combined physics loss.
        """
        loss_energy = self.energy_conservation(met_conc, met_next)
        loss_mass = self.mass_balance(flux)
        loss_stable = self.stability(met_conc, met_next)
        loss_smooth = self.smoothness(met_conc, met_next)
        loss_ec = self.energy_charge_valid(met_next)
        
        total = (
            1.0 * loss_energy +
            0.1 * loss_mass +
            10.0 * loss_stable +
            0.5 * loss_smooth +
            5.0 * loss_ec
        )
        
        return total, {
            'energy': loss_energy.item(),
            'mass': loss_mass.item(),
            'stable': loss_stable.item(),
            'smooth': loss_smooth.item(),
            'ec': loss_ec.item(),
        }

# Find ATP/ADP/AMP indices
atp_idx = next((i for i, n in enumerate(met_names) if 'atp' in n.lower() and 'datp' not in n.lower()), 0)
adp_idx = next((i for i, n in enumerate(met_names) if 'adp' in n.lower() and 'dadp' not in n.lower()), 1)
amp_idx = next((i for i, n in enumerate(met_names) if 'amp' in n.lower() and 'damp' not in n.lower() and 'camp' not in n.lower()), 2)

print(f"ATP index: {atp_idx} ({met_names[atp_idx]})")
print(f"ADP index: {adp_idx} ({met_names[adp_idx]})")
print(f"AMP index: {amp_idx} ({met_names[amp_idx]})")
print("✅ PhysicsLoss defined")

In [ ]:
#@title 5️⃣ DarkManifoldV6 - Hybrid Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class DarkManifoldV6(nn.Module):
    """
    Hybrid Biochemical Neural Model.
    
    FIXED (real biochemistry):
    - Stoichiometry matrix S
    - Known Km values
    - Michaelis-Menten equation
    
    LEARNED (neural network):
    - Unknown Km values
    - Gene → Enzyme mapping
    - Regulatory effects
    """
    
    def __init__(self, n_genes, n_metabolites, n_reactions, S, G, 
                 known_km_indices=None, known_km_values=None, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_metabolites = n_metabolites
        self.n_reactions = n_reactions
        
        # FIXED: Stoichiometry (real biochemistry)
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        # Substrate mask (which metabolites are substrates for each reaction)
        substrate_mask = (self.S < 0).float()
        self.register_buffer('substrate_mask', substrate_mask)
        
        # FIXED: Known Km values (from BRENDA)
        # Initialize all Km to 1.0, then fix known ones
        self.log_Km = nn.Parameter(torch.zeros(n_reactions))
        
        # Create mask for known vs unknown Km
        km_is_known = torch.zeros(n_reactions, dtype=torch.bool)
        if known_km_indices is not None:
            # For each known metabolite, fix Km for reactions where it's a substrate
            for met_idx, km_val in zip(known_km_indices, known_km_values):
                # Find reactions where this metabolite is a substrate
                rxn_mask = S[met_idx, :] < 0
                with torch.no_grad():
                    self.log_Km.data[rxn_mask] = np.log(km_val)
                km_is_known = km_is_known | torch.tensor(rxn_mask)
        
        self.register_buffer('km_is_known', km_is_known)
        print(f"  Fixed Km: {km_is_known.sum().item()} reactions")
        print(f"  Learnable Km: {(~km_is_known).sum().item()} reactions")
        
        # LEARNED: Gene → Enzyme mapping
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
            nn.Softplus(),  # Vmax > 0
        )
        
        # LEARNED: Regulatory network
        # Which metabolites regulate which reactions
        self.regulation_net = nn.Sequential(
            nn.Linear(n_metabolites, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
            nn.Tanh(),  # Output in [-1, 1] for activation/inhibition
        )
        
        # Learnable integration time scale (per-reaction)
        self.log_tau = nn.Parameter(torch.zeros(n_reactions) * 0.01)
    
    @property
    def Km(self):
        # Km values, clamped to reasonable range
        return torch.exp(self.log_Km).clamp(0.001, 100.0)
    
    @property
    def tau(self):
        return torch.exp(self.log_tau).clamp(0.1, 10.0)
    
    def forward(self, gene_expr, met_conc, dt=0.01):
        """
        Forward pass implementing real biochemistry.
        """
        batch_size = gene_expr.shape[0]
        
        # ===== STEP 1: Gene → Enzyme (LEARNED) =====
        Vmax = self.gene_encoder(gene_expr)  # (B, n_rxns)
        
        # ===== STEP 2: Compute substrate concentrations =====
        # For each reaction, get concentration of its substrates
        # substrate_mask: (n_mets, n_rxns), 1 where S < 0
        n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)  # (n_rxns,)
        substrate_conc = (met_conc @ self.substrate_mask) / n_substrates  # (B, n_rxns)
        substrate_conc = substrate_conc + 0.001  # Avoid division by zero
        
        # ===== STEP 3: Michaelis-Menten kinetics (FIXED equation) =====
        # v = Vmax * S / (Km + S)
        Km = self.Km  # (n_rxns,)
        mm_term = substrate_conc / (Km + substrate_conc)  # (B, n_rxns)
        
        # ===== STEP 4: Regulatory effects (LEARNED) =====
        # reg in [-1, 1]: negative = inhibition, positive = activation
        reg = self.regulation_net(met_conc)  # (B, n_rxns)
        reg_factor = 1.0 + 0.5 * reg  # Scale to [0.5, 1.5]
        
        # ===== STEP 5: Compute flux =====
        flux = Vmax * mm_term * reg_factor * self.tau  # (B, n_rxns)
        
        # ===== STEP 6: Stoichiometry constraint (FIXED) =====
        # dM/dt = S @ flux
        dM_dt = flux @ self.S.T  # (B, n_mets)
        
        # ===== STEP 7: Euler integration =====
        met_next = met_conc + dt * dM_dt
        met_next = met_next.clamp(min=0)  # Concentrations >= 0
        
        return {
            'next_metabolites': met_next,
            'flux': flux,
            'dM_dt': dM_dt,
            'Vmax': Vmax,
            'regulation': reg,
            'substrate_conc': substrate_conc,
        }
    
    def simulate(self, gene_expr, met_init, n_steps=100, dt=0.01):
        """
        Run simulation forward.
        """
        traj = [met_init]
        met = met_init
        
        for _ in range(n_steps):
            out = self.forward(gene_expr, met, dt)
            met = out['next_metabolites']
            traj.append(met)
        
        return torch.stack(traj, dim=1)  # (B, T+1, n_mets)
    
    def knockout(self, gene_expr, gene_idx):
        """Zero out a gene."""
        ko = gene_expr.clone()
        ko[:, gene_idx] = 0.0
        return ko


model = DarkManifoldV6(
    n_genes=n_genes,
    n_metabolites=n_mets,
    n_reactions=n_rxns,
    S=S,
    G=G,
    known_km_indices=known_km_indices,
    known_km_values=known_km_values,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV6 (Hybrid):")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title 6️⃣ Train with Physics-Based Losses

n_epochs = 2000
batch_size = 32
n_steps = 50  # Steps per trajectory
lr = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

physics_loss = PhysicsLoss(S, atp_idx, adp_idx, amp_idx).to(device)

# Build initial concentration tensor
init_conc = torch.ones(n_mets) * 0.5  # Default 0.5 mM
for i, name in enumerate(met_names):
    name_lower = name.lower()
    for key, val in INITIAL_CONC.items():
        if key in name_lower:
            init_conc[i] = val
            break
init_conc = init_conc.to(device)

def train_step():
    model.train()
    
    # Random initial conditions (around known values)
    gene_expr = torch.rand(batch_size, n_genes, device=device) * 2.0 + 0.5
    met_conc = init_conc.unsqueeze(0).expand(batch_size, -1) * (0.8 + 0.4 * torch.rand(batch_size, n_mets, device=device))
    
    total_loss = 0
    all_metrics = {'energy': 0, 'mass': 0, 'stable': 0, 'smooth': 0, 'ec': 0}
    
    # Simulate forward and accumulate physics losses
    for _ in range(n_steps):
        out = model(gene_expr, met_conc)
        
        loss, metrics = physics_loss(met_conc, out['next_metabolites'], out['flux'])
        total_loss += loss
        
        for k in all_metrics:
            all_metrics[k] += metrics[k]
        
        met_conc = out['next_metabolites'].detach()  # Don't backprop through entire trajectory
    
    # Average
    total_loss = total_loss / n_steps
    for k in all_metrics:
        all_metrics[k] /= n_steps
    
    return total_loss, all_metrics

losses = []
print(f"Training v6: {n_epochs} epochs")
print(f"Physics-based losses: energy conservation, mass balance, stability")
print(f"No external generator - the model IS the simulation")
print()

for epoch in tqdm(range(n_epochs)):
    optimizer.zero_grad()
    loss, metrics = train_step()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 400 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f} | "
              f"energy={metrics['energy']:.4f} stable={metrics['stable']:.4f} ec={metrics['ec']:.4f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
#@title 7️⃣ Evaluate: Run Simulation
import matplotlib.pyplot as plt

model.eval()

# Run simulation from initial conditions
gene_expr = torch.ones(1, n_genes, device=device)  # Normal expression
met_init = init_conc.unsqueeze(0)

with torch.no_grad():
    traj = model.simulate(gene_expr, met_init, n_steps=200, dt=0.01)

traj_np = traj[0].cpu().numpy()  # (T, n_mets)

# Plot key metabolites
fig, axes = plt.subplots(2, 3, figsize=(12, 6))

key_mets = [
    (atp_idx, 'ATP'),
    (adp_idx, 'ADP'),
    (amp_idx, 'AMP'),
]

# Find some glycolysis intermediates
for i, name in enumerate(met_names):
    if 'glucose' in name.lower() and 'phosphate' not in name.lower():
        key_mets.append((i, name))
        break
for i, name in enumerate(met_names):
    if 'pyruvate' in name.lower():
        key_mets.append((i, name))
        break
for i, name in enumerate(met_names):
    if 'lactate' in name.lower():
        key_mets.append((i, name))
        break

for ax, (idx, name) in zip(axes.flatten(), key_mets[:6]):
    ax.plot(traj_np[:, idx], 'b-', linewidth=2)
    ax.set_title(name)
    ax.set_xlabel('Time step')
    ax.set_ylabel('Concentration (mM)')

# Energy charge over time
atp_traj = traj_np[:, atp_idx]
adp_traj = traj_np[:, adp_idx]
amp_traj = traj_np[:, amp_idx]
ec_traj = (atp_traj + 0.5 * adp_traj) / (atp_traj + adp_traj + amp_traj + 1e-6)

plt.suptitle('v6 Hybrid: Self-Simulated Trajectories')
plt.tight_layout()
plt.savefig('trajectory_v6.png', dpi=150)
plt.show()

# Energy charge plot
plt.figure(figsize=(8, 3))
plt.plot(ec_traj, 'g-', linewidth=2)
plt.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Min healthy')
plt.axhline(y=0.95, color='r', linestyle='--', alpha=0.5, label='Max healthy')
plt.xlabel('Time step')
plt.ylabel('Energy Charge')
plt.title('Energy Charge (should stay 0.8-0.95)')
plt.legend()
plt.tight_layout()
plt.savefig('energy_charge_v6.png', dpi=150)
plt.show()

print(f"\nEnergy charge: {ec_traj[0]:.3f} → {ec_traj[-1]:.3f}")
print(f"ATP: {atp_traj[0]:.2f} → {atp_traj[-1]:.2f} mM")

In [ ]:
#@title 8️⃣ Gene Knockout Analysis

print("="*60)
print("GENE KNOCKOUT ANALYSIS")
print("="*60)

# Baseline simulation
gene_wt = torch.ones(1, n_genes, device=device)
met_init = init_conc.unsqueeze(0)

with torch.no_grad():
    wt_traj = model.simulate(gene_wt, met_init, n_steps=200)

wt_final = wt_traj[0, -1].cpu()
wt_atp = wt_final[atp_idx].item()
wt_ec = (wt_final[atp_idx] + 0.5 * wt_final[adp_idx]) / (wt_final[atp_idx] + wt_final[adp_idx] + wt_final[amp_idx] + 1e-6)

print(f"Wildtype: ATP={wt_atp:.2f} mM, EC={wt_ec:.3f}")
print()

# Test each gene knockout
ko_results = []

for i in tqdm(range(n_genes)):
    gene_ko = model.knockout(gene_wt, i)
    
    with torch.no_grad():
        ko_traj = model.simulate(gene_ko, met_init, n_steps=200)
    
    ko_final = ko_traj[0, -1].cpu()
    ko_atp = ko_final[atp_idx].item()
    ko_ec = (ko_final[atp_idx] + 0.5 * ko_final[adp_idx]) / (ko_final[atp_idx] + ko_final[adp_idx] + ko_final[amp_idx] + 1e-6)
    
    # Viability: ATP > 50% of WT and EC > 0.7
    is_lethal = (ko_atp < 0.5 * wt_atp) or (ko_ec < 0.7)
    
    ko_results.append({
        'gene': genes[i],
        'atp': ko_atp,
        'ec': ko_ec.item(),
        'atp_ratio': ko_atp / wt_atp,
        'is_lethal': is_lethal,
    })

# Summary
lethal = [r for r in ko_results if r['is_lethal']]
viable = [r for r in ko_results if not r['is_lethal']]

print(f"\nResults:")
print(f"  Essential (lethal KO): {len(lethal)} ({100*len(lethal)/n_genes:.1f}%)")
print(f"  Non-essential: {len(viable)} ({100*len(viable)/n_genes:.1f}%)")

# Most lethal
sorted_ko = sorted(ko_results, key=lambda x: x['atp_ratio'])
print(f"\nMost lethal knockouts (lowest ATP):")
for r in sorted_ko[:10]:
    status = "❌ LETHAL" if r['is_lethal'] else "✓ viable"
    print(f"  {r['gene']}: ATP={r['atp']:.2f} ({r['atp_ratio']:.0%} of WT) EC={r['ec']:.3f} {status}")

In [ ]:
#@title 9️⃣ Inspect Learned Parameters

# Km distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

km_values = model.Km.detach().cpu().numpy()
km_known_mask = model.km_is_known.cpu().numpy()

axes[0].hist(km_values[km_known_mask], bins=20, alpha=0.7, label=f'Fixed ({km_known_mask.sum()})', color='teal')
axes[0].hist(km_values[~km_known_mask], bins=20, alpha=0.7, label=f'Learned ({(~km_known_mask).sum()})', color='coral')
axes[0].set_xlabel('Km value (mM)')
axes[0].set_ylabel('Count')
axes[0].set_title('Km Distribution')
axes[0].legend()

# Tau distribution
tau_values = model.tau.detach().cpu().numpy()
axes[1].hist(tau_values, bins=30, alpha=0.7, color='purple')
axes[1].set_xlabel('Tau value')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Tau Distribution (std={tau_values.std():.3f})')

plt.tight_layout()
plt.savefig('params_v6.png', dpi=150)
plt.show()

print(f"Km (fixed): mean={km_values[km_known_mask].mean():.3f}, std={km_values[km_known_mask].std():.3f}")
print(f"Km (learned): mean={km_values[~km_known_mask].mean():.3f}, std={km_values[~km_known_mask].std():.3f}")
print(f"Tau: mean={tau_values.mean():.3f}, std={tau_values.std():.3f}")

In [ ]:
#@title 🔟 Save Model

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'knockout_results': ko_results,
    'known_km_indices': known_km_indices,
    'known_km_values': known_km_values,
    'version': 'v6_hybrid',
    'approach': 'physics_where_known_learning_where_not',
    'metrics': {
        'n_essential': len(lethal),
        'n_non_essential': len(viable),
        'wt_atp': wt_atp,
        'wt_ec': wt_ec.item(),
        'km_learned_std': km_values[~km_known_mask].std(),
        'tau_std': tau_values.std(),
    }
}

torch.save(save_dict, 'dark_manifold_v6_hybrid.pt')

print("="*60)
print("DARK MANIFOLD v6 - SUMMARY")
print("="*60)
print(f"\nApproach: Physics where known, learning where not")
print(f"\nFixed components:")
print(f"  - Stoichiometry matrix (304×338)")
print(f"  - Known Km values ({len(known_km_indices)} metabolites)")
print(f"  - Michaelis-Menten equation")
print(f"\nLearned components:")
print(f"  - Unknown Km values")
print(f"  - Gene→Enzyme mapping")
print(f"  - Regulatory effects")
print(f"\nKnockout results:")
print(f"  - Essential genes: {len(lethal)}/{n_genes} ({100*len(lethal)/n_genes:.1f}%)")

from google.colab import files
files.download('dark_manifold_v6_hybrid.pt')
files.download('trajectory_v6.png')
files.download('energy_charge_v6.png')
files.download('params_v6.png')